####Paso 1 - Leer el archivo CSV usando "DataFrameReader de Spark"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configurations"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
genre_schema = StructType([
    StructField("genreId", IntegerType(), False),
    StructField("genreName", StringType(), True)
])

In [0]:
genre_df = spark.read \
    .option("header", "True") \
    .schema(genre_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/genre.csv")

####Paso 2 - Seleccionar sólo las columnas requeridas

In [0]:
from pyspark.sql.functions import col

In [0]:
genre_selected_df = genre_df.select(col("genreId"), col("genreName"))

####Paso 3 - Cambiar el nombre de las columnas según lo requerido

In [0]:
genre_renamed_df = genre_selected_df.withColumnRenamed("genreId", "genre_id").withColumnRenamed("genreName", "genre_name")

####Paso 4 - Agregar la columna "ingestion_date" y "enviroment" al dataframe

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
genre_final_df = genre_renamed_df.withColumns({
    "ingestion_date": current_timestamp(),
    "ingestion_source": lit("Produccion"),
    "file_date": lit(v_file_date)
})

###Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
#genre_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/genre")
genre_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.genre")

In [0]:
%sql
SELECT *
FROM movie_silver.genre;

In [0]:
#%fs
#ls abfss://silver@moviehistorybymg.dfs.core.windows.net/genre

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/genre"))

In [0]:
dbutils.notebook.exit("Success")